# Residual RRG × APC テーマバスケット戦略 — Standalone Notebook

このNotebookは、添付されていた `theme_basket_strategy.py`、`apc_strategy_variants.py`、および追加した residual RRG ロジックを**すべてNotebook内に展開**した自己完結版です。

外部 `.py` ファイルを import しません。`USE_SAMPLE_DATA=True` のまま実行するとサンプルデータで動作確認できます。実データで使う場合は、`USE_SAMPLE_DATA=False` に変更し、CSVパスを設定してください。

主な実行内容：

1. Level 1: 生テーマモメンタム
2. Level 2: ファクター残差テーマモメンタム
3. Level 3: 個別銘柄ベース APC / eigenshare coherence gate
4. Level 4: ニュース・ガバナンス・回転率 crowding overlay
5. APC strategy variants
6. **Residual RRG strategy**: 残差RS-Ratio、残差RS-Momentum、象限、RRGチャート、バックテスト


## 0. Imports

In [ ]:
from pathlib import Path
from dataclasses import replace
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 30)

## 1. Core implementation: theme basket strategy

このセルは `theme_basket_strategy.py` の内容をNotebook内に展開しています。

In [ ]:
"""Theme basket strategy helper module.

Implements a research-oriented version of:
Coherence-Gated Narrative Momentum with Governance/PBR Catalyst Overlay.

Dependencies: pandas, numpy.
"""
import json
import math
import os
from dataclasses import asdict, dataclass
from typing import Dict, Literal, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd

InputKind = Literal["price", "return", "auto"]
Mode = Literal["long_only", "long_short"]


@dataclass
class StrategyConfig:
    theme_input_kind: InputKind = "auto"
    stock_input_kind: InputKind = "auto"

    residualize_themes: bool = True
    residualize_stocks: bool = True
    rolling_beta_window: int = 252
    rolling_beta_min_obs: int = 126
    ridge_lambda: float = 1e-4

    mom_lookbacks: Tuple[int, int, int] = (252, 126, 63)
    mom_weights: Tuple[float, float, float] = (0.50, 0.30, 0.20)
    mom_skip: int = 21
    mom_vol_floor: float = 1e-4

    use_coherence_gate: bool = True
    coherence_window: int = 126
    coherence_min_names: int = 8
    coherence_min_effective_names: float = 5.0
    coherence_apc_threshold: float = 0.0
    coherence_score_threshold: Optional[float] = None
    apc_weight: float = 0.65
    eigenshare_weight: float = 0.35

    use_news: bool = True
    use_governance: bool = True
    use_leadlag: bool = False
    use_valuation: bool = False
    use_crowding: bool = True

    w_momentum: float = 0.45
    w_news: float = 0.20
    w_governance: float = 0.20
    w_leadlag: float = 0.10
    w_valuation: float = 0.05
    w_crowding_penalty: float = 0.25

    mode: Mode = "long_only"
    rebalance: str = "ME"
    top_n: Optional[int] = None
    bottom_n: Optional[int] = None
    long_quantile: float = 0.20
    short_quantile: float = 0.20
    max_theme_weight: float = 0.15
    min_abs_score: float = 0.0
    gross_exposure: float = 1.0
    net_exposure: float = 1.0

    cost_bps_per_turnover: float = 5.0
    annualization: int = 252
    min_theme_obs: int = 126
    fill_missing_returns: bool = False

    def normalized_mom_weights(self) -> np.ndarray:
        w = np.asarray(self.mom_weights, dtype=float)
        if not np.isfinite(w).all() or w.sum() == 0:
            raise ValueError("mom_weights must be finite and have non-zero sum")
        return w / w.sum()


def _infer_date_col(columns: Sequence[str]) -> str:
    for c in ["date", "Date", "DATE", "datetime", "timestamp", "Timestamp"]:
        if c in columns:
            return c
    return columns[0]


def read_matrix_csv(path: str, date_col: Optional[str] = None, value_col: Optional[str] = None, key_col: Optional[str] = None) -> pd.DataFrame:
    """Read wide or long time-series CSV into date-indexed matrix."""
    df = pd.read_csv(path)
    if df.empty:
        raise ValueError(f"Empty CSV: {path}")
    date_col = date_col or _infer_date_col(list(df.columns))
    if date_col not in df.columns:
        raise ValueError(f"date_col={date_col} not found in {path}")
    cols_lower = {str(c).lower(): c for c in df.columns}
    if key_col is None:
        for c in ["theme", "ticker", "symbol", "ric", "factor", "name"]:
            if c in cols_lower:
                key_col = cols_lower[c]
                break
    if value_col is None:
        for c in ["value", "price", "px", "return", "ret", "score", "weight"]:
            if c in cols_lower:
                value_col = cols_lower[c]
                break
    df[date_col] = pd.to_datetime(df[date_col])
    if key_col is not None and value_col is not None and key_col in df.columns and value_col in df.columns:
        out = df.pivot_table(index=date_col, columns=key_col, values=value_col, aggfunc="last")
        out = out.sort_index()
        out.columns = out.columns.astype(str)
        return out.apply(pd.to_numeric, errors="coerce")
    out = df.set_index(date_col).sort_index()
    out = out.loc[:, ~out.columns.astype(str).str.lower().isin(["unnamed: 0"])]
    out.columns = out.columns.astype(str)
    return out.apply(pd.to_numeric, errors="coerce")


def read_constituents_csv(path: str) -> pd.DataFrame:
    """Read PIT constituents. Required: theme, ticker/ric/symbol, weight."""
    df = pd.read_csv(path)
    cols = {str(c).lower(): c for c in df.columns}
    theme_col = cols.get("theme")
    ticker_col = cols.get("ticker") or cols.get("ric") or cols.get("symbol")
    weight_col = cols.get("weight") or cols.get("wgt")
    date_col = cols.get("date") or cols.get("effective_date")
    if theme_col is None or ticker_col is None or weight_col is None:
        raise ValueError("constituents_csv requires theme, ticker/ric/symbol, weight")
    out = pd.DataFrame({
        "date": pd.to_datetime(df[date_col]) if date_col else pd.Timestamp("1900-01-01"),
        "theme": df[theme_col].astype(str),
        "ticker": df[ticker_col].astype(str),
        "weight": pd.to_numeric(df[weight_col], errors="coerce"),
    })
    out["purity"] = pd.to_numeric(df[cols["purity"]], errors="coerce") if "purity" in cols else 1.0
    out["liquidity_adj"] = pd.to_numeric(df[cols["liquidity_adj"]], errors="coerce") if "liquidity_adj" in cols else 1.0
    out = out.dropna(subset=["date", "theme", "ticker", "weight"])
    out["weight"] = out["weight"].clip(lower=0)
    out["purity"] = out["purity"].fillna(1.0).clip(lower=0)
    out["liquidity_adj"] = out["liquidity_adj"].fillna(1.0).clip(lower=0)
    return out.sort_values(["theme", "date", "ticker"]).reset_index(drop=True)


def prices_to_returns(data: pd.DataFrame, kind: InputKind = "auto") -> pd.DataFrame:
    x = data.copy().sort_index().replace([np.inf, -np.inf], np.nan)
    if kind == "auto":
        med_abs = np.nanmedian(np.abs(x.to_numpy(dtype=float)))
        kind = "price" if med_abs > 2 else "return"
    if kind == "price":
        return x.pct_change(fill_method=None).replace([np.inf, -np.inf], np.nan)
    if kind == "return":
        return x
    raise ValueError("kind must be price, return, or auto")


def zscore_cross_sectional(x: pd.DataFrame, winsor: float = 5.0) -> pd.DataFrame:
    mu = x.mean(axis=1, skipna=True)
    sd = x.std(axis=1, skipna=True, ddof=0).replace(0.0, np.nan)
    z = x.sub(mu, axis=0).div(sd, axis=0)
    if winsor is not None:
        z = z.clip(-winsor, winsor)
    return z


def rolling_residualize(returns: pd.DataFrame, factors: Optional[pd.DataFrame], window: int = 252, min_obs: int = 126, ridge_lambda: float = 1e-4) -> pd.DataFrame:
    """One-step rolling residualization. Beta at t is estimated with data <= t-1."""
    y = returns.copy().sort_index().replace([np.inf, -np.inf], np.nan)
    if factors is None or factors.empty:
        return y
    X = factors.copy().sort_index().replace([np.inf, -np.inf], np.nan)
    idx = y.index.intersection(X.index)
    y = y.loc[idx]
    X = X.loc[idx]
    Xfull = np.column_stack([np.ones(len(idx)), X.to_numpy(dtype=float)])
    ymat = y.to_numpy(dtype=float)
    out = np.full_like(ymat, np.nan, dtype=float)
    k = Xfull.shape[1]
    ridge = np.eye(k) * ridge_lambda
    ridge[0, 0] = 0.0
    for t in range(len(idx)):
        start, end = max(0, t - window), t
        if end - start < min_obs:
            continue
        Xw_all = Xfull[start:end]
        Xt = Xfull[t]
        if not np.isfinite(Xt).all():
            continue
        finite_X = np.isfinite(Xw_all).all(axis=1)
        for j in range(ymat.shape[1]):
            yw_all = ymat[start:end, j]
            mask = finite_X & np.isfinite(yw_all)
            if mask.sum() < min_obs or not np.isfinite(ymat[t, j]):
                continue
            Xw = Xw_all[mask]
            yw = yw_all[mask]
            try:
                beta = np.linalg.solve(Xw.T @ Xw + ridge, Xw.T @ yw)
            except np.linalg.LinAlgError:
                beta = np.linalg.pinv(Xw.T @ Xw + ridge) @ Xw.T @ yw
            out[t, j] = ymat[t, j] - float(Xt @ beta)
    return pd.DataFrame(out, index=idx, columns=y.columns)


def rolling_momentum(returns: pd.DataFrame, lookbacks=(252,126,63), weights=(0.5,0.3,0.2), skip=21, vol_floor=1e-4) -> pd.DataFrame:
    ret = returns.copy().sort_index()
    w = np.asarray(weights, dtype=float)
    w = w / w.sum()
    raw = pd.DataFrame(0.0, index=ret.index, columns=ret.columns)
    for lb, ww in zip(lookbacks, w):
        shifted = ret.shift(skip)
        cum = shifted.rolling(lb, min_periods=max(20, lb // 2)).sum()
        vol = shifted.rolling(lb, min_periods=max(20, lb // 2)).std(ddof=0).replace(0, np.nan)
        raw = raw + ww * (cum / vol.clip(lower=vol_floor))
    return zscore_cross_sectional(raw)


def return_acceleration_crowding(returns: pd.DataFrame, short: int = 21, medium: int = 63) -> pd.DataFrame:
    r_short = returns.rolling(short, min_periods=max(5, short // 2)).sum()
    r_med_lag = returns.shift(short).rolling(medium, min_periods=max(10, medium // 2)).sum()
    return zscore_cross_sectional(r_short - r_med_lag)


def rolling_theme_turnover_crowding(turnover: pd.DataFrame, window: int = 60) -> pd.DataFrame:
    x = np.log1p(turnover.replace([np.inf, -np.inf], np.nan).clip(lower=0))
    surpr = x - x.rolling(window, min_periods=max(10, window // 2)).mean()
    return zscore_cross_sectional(surpr)


def make_rebalance_dates(index: pd.DatetimeIndex, rule: str = "ME") -> pd.DatetimeIndex:
    s = pd.Series(np.arange(len(index)), index=index)
    try:
        vals = s.resample(rule).last().dropna().astype(int).values
    except ValueError:
        if rule == "ME":
            vals = s.resample("M").last().dropna().astype(int).values
        elif rule == "M":
            vals = s.resample("ME").last().dropna().astype(int).values
        else:
            raise
    return pd.DatetimeIndex(index[vals])


def effective_n(weights: Union[pd.Series, np.ndarray]) -> float:
    w = np.asarray(weights, dtype=float)
    w = w[np.isfinite(w)]
    if w.size == 0 or w.sum() <= 0:
        return np.nan
    w = w / w.sum()
    return float(1.0 / np.sum(w ** 2))


def _latest_constituents_for_date(constituents: pd.DataFrame, dt: pd.Timestamp) -> Dict[str, pd.DataFrame]:
    c = constituents[constituents["date"] <= dt]
    out = {}
    if c.empty:
        return out
    for theme, g in c.groupby("theme"):
        gg = g[g["date"] == g["date"].max()].copy()
        gg = gg[gg["weight"] > 0]
        if gg.empty:
            continue
        adj = gg["weight"] * gg["purity"].fillna(1) * gg["liquidity_adj"].fillna(1)
        if adj.sum() <= 0:
            continue
        gg["adj_weight"] = adj / adj.sum()
        out[str(theme)] = gg
    return out


def compute_coherence_features(residual_stock_returns: pd.DataFrame, constituents: pd.DataFrame, theme_columns: Sequence[str], signal_dates: Sequence[pd.Timestamp], window=126, min_names=8, min_effective_names=5.0):
    stock_ret = residual_stock_returns.sort_index()
    dates = pd.DatetimeIndex(signal_dates)
    themes = list(map(str, theme_columns))
    apc = pd.DataFrame(np.nan, index=dates, columns=themes)
    eig = pd.DataFrame(np.nan, index=dates, columns=themes)
    eff = pd.DataFrame(np.nan, index=dates, columns=themes)
    gate = pd.DataFrame(False, index=dates, columns=themes)
    for dt in dates:
        prev = stock_ret.index[stock_ret.index <= dt]
        if len(prev) == 0:
            continue
        dtx = prev[-1]
        loc = stock_ret.index.get_loc(dtx)
        if not isinstance(loc, (int, np.integer)):
            loc = int(np.asarray(loc).ravel()[-1])
        win = stock_ret.iloc[max(0, loc - window + 1):loc + 1]
        latest = _latest_constituents_for_date(constituents, dt)
        for theme in themes:
            gg = latest.get(theme)
            if gg is None:
                continue
            tickers = [t for t in gg["ticker"].astype(str).tolist() if t in win.columns]
            if len(tickers) < min_names:
                continue
            w = gg.set_index("ticker").loc[tickers, "adj_weight"]
            en = effective_n(w.values)
            eff.loc[dt, theme] = en
            if not np.isfinite(en) or en < min_effective_names:
                continue
            sub = win[tickers].dropna(axis=1, thresh=max(20, window // 2))
            if sub.shape[1] < min_names:
                continue
            corr = sub.corr(min_periods=max(20, window // 2)).to_numpy(dtype=float)
            iu = np.triu_indices_from(corr, k=1)
            pc = corr[iu]
            pc = pc[np.isfinite(pc)]
            if len(pc) == 0:
                continue
            apc.loc[dt, theme] = float(np.mean(pc))
            cmat = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
            np.fill_diagonal(cmat, 1.0)
            try:
                vals = np.clip(np.linalg.eigvalsh(cmat), 0, None)
                eig.loc[dt, theme] = float(vals[-1] / vals.sum()) if vals.sum() > 0 else np.nan
            except np.linalg.LinAlgError:
                pass
            gate.loc[dt, theme] = True
    return apc, eig, eff, gate


def align_feature_matrix(feature: Optional[pd.DataFrame], index: pd.Index, columns: Sequence[str], fill_value=0.0) -> pd.DataFrame:
    if feature is None or feature.empty:
        return pd.DataFrame(fill_value, index=index, columns=columns, dtype=float)
    return feature.sort_index().reindex(index).ffill().reindex(columns=columns).fillna(fill_value).astype(float)


def cap_and_normalize_long(w: pd.Series, max_weight: float, gross: float = 1.0) -> pd.Series:
    x = w.clip(lower=0).replace([np.inf, -np.inf], np.nan).fillna(0).astype(float)
    if x.sum() <= 0:
        return x
    x = x / x.sum() * gross
    cap = max_weight * gross if max_weight and max_weight <= 1 else max_weight
    if not cap or cap <= 0:
        return x
    for _ in range(20):
        over = x > cap
        if not over.any():
            break
        excess = float((x[over] - cap).sum())
        x[over] = cap
        under = ~over
        if x[under].sum() <= 0 or excess <= 1e-12:
            break
        x[under] += x[under] / x[under].sum() * excess
    return x / x.sum() * gross if x.sum() > 0 else x


def construct_theme_weights(scores: pd.DataFrame, config: StrategyConfig, gates: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    s = scores.replace([np.inf, -np.inf], np.nan).copy()
    if gates is not None:
        s = s.where(gates.reindex_like(s).fillna(False))
    weights = pd.DataFrame(0.0, index=s.index, columns=s.columns)
    n_cols = len(s.columns)
    for dt, row in s.iterrows():
        row = row.dropna()
        if row.empty:
            continue
        row = row[row.abs() >= config.min_abs_score]
        if config.mode == "long_only":
            r = row[row > 0]
            if r.empty:
                continue
            q = config.top_n if config.top_n is not None else max(1, int(math.ceil(n_cols * config.long_quantile)))
            sel = r.nlargest(q)
            weights.loc[dt, sel.index] = cap_and_normalize_long(sel, config.max_theme_weight, config.gross_exposure)
        else:
            ql = config.top_n if config.top_n is not None else max(1, int(math.ceil(n_cols * config.long_quantile)))
            qs = config.bottom_n if config.bottom_n is not None else max(1, int(math.ceil(n_cols * config.short_quantile)))
            longs = row[row > 0].nlargest(ql)
            shorts = row[row < 0].nsmallest(qs)
            if not longs.empty:
                wl = cap_and_normalize_long(longs, config.max_theme_weight, config.gross_exposure / 2)
                weights.loc[dt, wl.index] = wl
            if not shorts.empty:
                ws = cap_and_normalize_long(-shorts, config.max_theme_weight, config.gross_exposure / 2)
                weights.loc[dt, ws.index] = -ws
    return weights


def backtest_theme_weights(theme_returns: pd.DataFrame, weights_daily: pd.DataFrame, cost_bps_per_turnover: float = 5.0):
    ret = theme_returns.reindex_like(weights_daily).fillna(0)
    wlag = weights_daily.shift(1).fillna(0)
    gross = (wlag * ret).sum(axis=1)
    turnover = weights_daily.diff().abs().sum(axis=1).fillna(weights_daily.abs().sum(axis=1))
    cost = turnover * cost_bps_per_turnover / 10000.0
    net = gross - cost
    details = pd.DataFrame({"gross_return": gross, "turnover": turnover, "cost": cost, "net_return": net, "gross_exposure": weights_daily.abs().sum(axis=1), "net_exposure": weights_daily.sum(axis=1)})
    return net, details


def performance_metrics(returns: pd.Series, annualization: int = 252) -> pd.Series:
    r = returns.dropna().astype(float)
    if r.empty:
        return pd.Series(dtype=float)
    nav = (1 + r).cumprod()
    dd = nav / nav.cummax() - 1
    vol = r.std(ddof=0) * math.sqrt(annualization)
    ann_ret = nav.iloc[-1] ** (annualization / max(1, len(r))) - 1
    downside = r[r < 0].std(ddof=0) * math.sqrt(annualization)
    return pd.Series({
        "ann_return": ann_ret,
        "ann_vol": vol,
        "sharpe": ann_ret / vol if vol > 0 else np.nan,
        "sortino": ann_ret / downside if downside > 0 else np.nan,
        "max_drawdown": dd.min(),
        "calmar": ann_ret / abs(dd.min()) if dd.min() < 0 else np.nan,
        "hit_rate_daily": (r > 0).mean(),
        "avg_daily_return": r.mean(),
        "daily_vol": r.std(ddof=0),
        "skew": r.skew(),
        "kurtosis": r.kurtosis(),
        "n_obs": len(r),
        "final_nav": nav.iloc[-1],
    })


class ThemeBasketStrategy:
    def __init__(self, theme_data: pd.DataFrame, config: Optional[StrategyConfig] = None, factor_returns: Optional[pd.DataFrame] = None, constituents: Optional[pd.DataFrame] = None, stock_data: Optional[pd.DataFrame] = None, news_score: Optional[pd.DataFrame] = None, governance_score: Optional[pd.DataFrame] = None, leadlag_score: Optional[pd.DataFrame] = None, valuation_score: Optional[pd.DataFrame] = None, turnover_data: Optional[pd.DataFrame] = None):
        self.config = config or StrategyConfig()
        self.theme_raw = theme_data.copy().sort_index()
        self.factor_returns = factor_returns.copy().sort_index() if factor_returns is not None else None
        self.constituents = constituents
        self.stock_raw = stock_data.copy().sort_index() if stock_data is not None else None
        self.news_score = news_score
        self.governance_score = governance_score
        self.leadlag_score = leadlag_score
        self.valuation_score = valuation_score
        self.turnover_data = turnover_data
        self.results = {}

    def prepare_returns(self):
        cfg = self.config
        theme_returns = prices_to_returns(self.theme_raw, cfg.theme_input_kind)
        valid = theme_returns.columns[theme_returns.notna().sum() >= cfg.min_theme_obs]
        theme_returns = theme_returns[valid]
        if cfg.fill_missing_returns:
            theme_returns = theme_returns.fillna(0)
        stock_returns = None
        if self.stock_raw is not None:
            stock_returns = prices_to_returns(self.stock_raw, cfg.stock_input_kind)
            if cfg.fill_missing_returns:
                stock_returns = stock_returns.fillna(0)
        self.results["theme_returns"] = theme_returns
        if stock_returns is not None:
            self.results["stock_returns"] = stock_returns
        return theme_returns, stock_returns

    def compute(self):
        cfg = self.config
        theme_returns, stock_returns = self.prepare_returns()
        idx, cols = theme_returns.index, list(theme_returns.columns)
        factors = self.factor_returns.reindex(idx).astype(float) if self.factor_returns is not None else None

        if cfg.residualize_themes and factors is not None and not factors.empty:
            theme_resid = rolling_residualize(theme_returns, factors, cfg.rolling_beta_window, cfg.rolling_beta_min_obs, cfg.ridge_lambda)
        else:
            theme_resid = theme_returns.copy()
        self.results["theme_residual_returns"] = theme_resid
        mom = rolling_momentum(theme_resid, cfg.mom_lookbacks, cfg.normalized_mom_weights(), cfg.mom_skip, cfg.mom_vol_floor)
        self.results["momentum_score"] = mom

        coherence_gate = pd.DataFrame(True, index=idx, columns=cols)
        coherence_score = pd.DataFrame(0.0, index=idx, columns=cols)
        apc_full = pd.DataFrame(np.nan, index=idx, columns=cols)
        eig_full = pd.DataFrame(np.nan, index=idx, columns=cols)
        eff_full = pd.DataFrame(np.nan, index=idx, columns=cols)

        if cfg.use_coherence_gate and self.constituents is not None and stock_returns is not None:
            stock_factors = self.factor_returns.reindex(stock_returns.index).astype(float) if self.factor_returns is not None else None
            if cfg.residualize_stocks and stock_factors is not None and not stock_factors.empty:
                stock_resid = rolling_residualize(stock_returns, stock_factors, cfg.rolling_beta_window, cfg.rolling_beta_min_obs, cfg.ridge_lambda)
            else:
                stock_resid = stock_returns.copy()
            self.results["stock_residual_returns"] = stock_resid
            rebal_dates = make_rebalance_dates(idx, cfg.rebalance)
            apc, eig, eff, gate_rebal = compute_coherence_features(stock_resid, self.constituents, cols, rebal_dates, cfg.coherence_window, cfg.coherence_min_names, cfg.coherence_min_effective_names)
            apc_full = apc.reindex(idx).ffill()
            eig_full = eig.reindex(idx).ffill()
            eff_full = eff.reindex(idx).ffill()
            coherence_score = (cfg.apc_weight * zscore_cross_sectional(apc_full) + cfg.eigenshare_weight * zscore_cross_sectional(eig_full)).fillna(0)
            hard_gate = (apc_full > cfg.coherence_apc_threshold) & (eff_full >= cfg.coherence_min_effective_names)
            if cfg.coherence_score_threshold is not None:
                hard_gate &= coherence_score >= cfg.coherence_score_threshold
            coherence_gate = hard_gate.fillna(False)
        elif cfg.use_coherence_gate:
            self.results["warning_coherence"] = pd.Series(["Coherence gate requested but constituents and/or stock_data were not supplied. Using all-True gate."])

        self.results["coherence_apc"] = apc_full
        self.results["coherence_eigenshare"] = eig_full
        self.results["coherence_effective_n"] = eff_full
        self.results["coherence_score"] = coherence_score
        self.results["coherence_gate"] = coherence_gate

        news = align_feature_matrix(self.news_score, idx, cols, 0.0)
        gov = align_feature_matrix(self.governance_score, idx, cols, 0.0)
        leadlag = align_feature_matrix(self.leadlag_score, idx, cols, 0.0)
        valuation = align_feature_matrix(self.valuation_score, idx, cols, 0.0)
        crowd = return_acceleration_crowding(theme_returns)
        if self.turnover_data is not None:
            turn = align_feature_matrix(self.turnover_data, idx, cols, np.nan)
            crowd = zscore_cross_sectional(0.5 * crowd + 0.5 * rolling_theme_turnover_crowding(turn))
        crowd = crowd.fillna(0)
        self.results["news_score"] = news
        self.results["governance_score"] = gov
        self.results["leadlag_score"] = leadlag
        self.results["valuation_score"] = valuation
        self.results["crowding_score"] = crowd

        score = cfg.w_momentum * mom.fillna(0)
        if cfg.use_news:
            score += cfg.w_news * zscore_cross_sectional(news).fillna(0)
        if cfg.use_governance:
            score += cfg.w_governance * zscore_cross_sectional(gov).fillna(0)
        if cfg.use_leadlag:
            score += cfg.w_leadlag * zscore_cross_sectional(leadlag).fillna(0)
        if cfg.use_valuation:
            score += cfg.w_valuation * zscore_cross_sectional(valuation).fillna(0)
        if cfg.use_crowding:
            short_ret = theme_returns.rolling(10, min_periods=5).sum()
            break_mask = (short_ret < 0).astype(float)
            score -= cfg.w_crowding_penalty * crowd * (0.5 + 0.5 * break_mask)

        score_gated = score.where(coherence_gate)
        self.results["final_score"] = score
        self.results["final_score_gated"] = score_gated

        rebal_dates = make_rebalance_dates(idx, cfg.rebalance)
        w_rebal = construct_theme_weights(score_gated.reindex(rebal_dates), cfg, gates=coherence_gate.reindex(rebal_dates).fillna(False))
        w_daily = w_rebal.reindex(idx).ffill().fillna(0)
        strat_ret, details = backtest_theme_weights(theme_returns, w_daily, cfg.cost_bps_per_turnover)
        metrics = performance_metrics(strat_ret, cfg.annualization)
        self.results["theme_weights_rebalance"] = w_rebal
        self.results["theme_weights_daily"] = w_daily
        self.results["strategy_returns"] = strat_ret
        self.results["backtest_details"] = details
        self.results["metrics"] = metrics
        return self.results

    def save_results(self, output_dir: str):
        os.makedirs(output_dir, exist_ok=True)
        for name, obj in self.results.items():
            path = os.path.join(output_dir, f"{name}.csv")
            if isinstance(obj, pd.DataFrame):
                obj.to_csv(path, index=True)
            elif isinstance(obj, pd.Series):
                obj.to_csv(path, header=True)
        with open(os.path.join(output_dir, "config.json"), "w", encoding="utf-8") as f:
            json.dump(asdict(self.config), f, ensure_ascii=False, indent=2)


## 2. APC strategy variants implementation

このセルは `apc_strategy_variants.py` の内容をNotebook内に展開しています。

In [ ]:
"""APC / residual-coherence strategy variants.

This add-on module assumes that Level 3 (Coherence-gated) has already been
run with ``theme_basket_strategy.py`` and that ``results_l3`` contains:

- theme_returns
- momentum_score
- coherence_apc
- coherence_effective_n
- coherence_gate

Implemented variants
--------------------
1. Pure Coherence Strategy
   Buy themes with high time-series-z-scored APC, cross-sectionally relativized.

2. Coherence-Gated Momentum
   Buy positive residual momentum themes that pass the coherence gate.

3. APC Activation + Positive Momentum
   Buy high APC activation score themes, but only if residual momentum is positive.

Dependencies: pandas, numpy, and theme_basket_strategy.py
"""
import os
from dataclasses import replace
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd

# Core strategy classes/functions are defined above in this notebook.


def rolling_time_series_zscore(
    x: pd.DataFrame,
    window: int = 252,
    min_periods: Optional[int] = None,
    std_floor: float = 1e-8,
    winsor: Optional[float] = 5.0,
) -> pd.DataFrame:
    """Rolling time-series z-score for each column.

    Parameters
    ----------
    x:
        Date x theme matrix, e.g. APC.
    window:
        Rolling window length in trading days.
    min_periods:
        Minimum observations. If omitted, max(20, window//2).
    std_floor:
        Std floor to avoid division by zero.
    winsor:
        If not None, clip z-scores to [-winsor, winsor].

    Notes
    -----
    The z-score at date t uses the rolling mean/std including t. In the
    provided backtest engine weights are shifted by one day, so this does not
    use t's signal to earn t's return.
    """
    if min_periods is None:
        min_periods = max(20, window // 2)
    y = x.copy().sort_index().replace([np.inf, -np.inf], np.nan)
    mu = y.rolling(window=window, min_periods=min_periods).mean()
    sd = y.rolling(window=window, min_periods=min_periods).std(ddof=0)
    z = (y - mu) / sd.clip(lower=std_floor)
    if winsor is not None:
        z = z.clip(-winsor, winsor)
    return z


def make_apc_activation_score(
    apc: pd.DataFrame,
    ts_window: int = 252,
    ts_min_periods: Optional[int] = None,
    winsor: Optional[float] = 5.0,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Create APC activation score.

    Step 1: time-series z-score by theme:
        How unusual is the theme's APC versus its own history?

    Step 2: cross-sectional z-score at each date:
        How unusual is that activation versus other themes today?

    Returns
    -------
    apc_ts_z, apc_ts_cs_z
    """
    apc_ts_z = rolling_time_series_zscore(
        apc,
        window=ts_window,
        min_periods=ts_min_periods,
        winsor=winsor,
    )
    apc_ts_cs_z = zscore_cross_sectional(apc_ts_z, winsor=winsor)
    return apc_ts_z, apc_ts_cs_z


def _eligibility_from_results(
    results_l3: Dict[str, object],
    min_effective_n: float = 5.0,
    use_hard_gate: bool = True,
) -> pd.DataFrame:
    """Build strategy eligibility mask from Level 3 outputs."""
    apc = results_l3["coherence_apc"]
    eligibility = apc.notna()

    eff = results_l3.get("coherence_effective_n")
    if isinstance(eff, pd.DataFrame):
        eligibility &= eff >= min_effective_n

    if use_hard_gate:
        gate = results_l3.get("coherence_gate")
        if isinstance(gate, pd.DataFrame):
            eligibility &= gate.reindex_like(eligibility).fillna(False)

    return eligibility.fillna(False)


def build_apc_variant_scores(
    results_l3: Dict[str, object],
    apc_z_window: int = 252,
    apc_z_min_periods: Optional[int] = None,
    min_effective_n: float = 5.0,
    use_hard_gate: bool = True,
    winsor: Optional[float] = 5.0,
) -> Dict[str, pd.DataFrame]:
    """Build signal matrices for the three requested APC strategies.

    Returns a dict containing:
    - apc_ts_z
    - apc_ts_cs_z
    - pure_coherence_score
    - coherence_gated_momentum_score
    - apc_activation_momentum_positive_score
    - eligibility
    """
    required = ["coherence_apc", "momentum_score"]
    for key in required:
        if key not in results_l3:
            raise KeyError(f"results_l3 must contain {key!r}")

    apc = results_l3["coherence_apc"]
    mom = results_l3["momentum_score"].reindex_like(apc)

    apc_ts_z, apc_ts_cs_z = make_apc_activation_score(
        apc,
        ts_window=apc_z_window,
        ts_min_periods=apc_z_min_periods,
        winsor=winsor,
    )

    eligibility = _eligibility_from_results(
        results_l3,
        min_effective_n=min_effective_n,
        use_hard_gate=use_hard_gate,
    ).reindex_like(apc_ts_cs_z).fillna(False)

    # 1. Pure Coherence Strategy:
    #    rank by APC activation score. Direction of return is NOT used.
    pure = apc_ts_cs_z.where(eligibility)

    # 2. Coherence-Gated Momentum:
    #    rank by residual momentum, but only among coherent themes.
    gated_mom = mom.where(eligibility)

    # 3. APC Activation + Positive Momentum:
    #    rank by APC activation score, but require residual momentum > 0.
    apc_pos_mom = apc_ts_cs_z.where(eligibility & (mom > 0))

    return {
        "apc_ts_z": apc_ts_z,
        "apc_ts_cs_z": apc_ts_cs_z,
        "pure_coherence_score": pure,
        "coherence_gated_momentum_score": gated_mom,
        "apc_activation_momentum_positive_score": apc_pos_mom,
        "eligibility": eligibility,
    }


def backtest_score_variant(
    theme_returns: pd.DataFrame,
    score: pd.DataFrame,
    config: StrategyConfig,
    gate: Optional[pd.DataFrame] = None,
    cost_bps_per_turnover: Optional[float] = None,
) -> Dict[str, object]:
    """Turn a score matrix into rebalance weights and backtest returns."""
    rebal_dates = make_rebalance_dates(theme_returns.index, config.rebalance)
    score_rebal = score.reindex(rebal_dates)
    gate_rebal = gate.reindex(rebal_dates).fillna(False) if gate is not None else None

    weights_rebal = construct_theme_weights(score_rebal, config, gates=gate_rebal)
    weights_daily = weights_rebal.reindex(theme_returns.index).ffill().fillna(0.0)

    strat_ret, details = backtest_theme_weights(
        theme_returns=theme_returns,
        weights_daily=weights_daily,
        cost_bps_per_turnover=cost_bps_per_turnover
        if cost_bps_per_turnover is not None
        else config.cost_bps_per_turnover,
    )
    metrics = performance_metrics(strat_ret, config.annualization)

    return {
        "weights_rebalance": weights_rebal,
        "weights_daily": weights_daily,
        "returns": strat_ret,
        "details": details,
        "metrics": metrics,
    }


def run_apc_strategy_variants(
    results_l3: Dict[str, object],
    config: Optional[StrategyConfig] = None,
    apc_z_window: int = 252,
    apc_z_min_periods: Optional[int] = None,
    min_effective_n: float = 5.0,
    use_hard_gate: bool = True,
    winsor: Optional[float] = 5.0,
    output_dir: Optional[str] = None,
) -> Dict[str, object]:
    """Run the three requested APC strategy variants.

    Parameters
    ----------
    results_l3:
        Output dict from Level 3 strategy run.
    config:
        Portfolio config. If None, uses StrategyConfig with long-only defaults.
        For these variants, ``mode='long_only'`` is strongly recommended.
    apc_z_window:
        Time-series z-score window for APC.
    apc_z_min_periods:
        Minimum observations for APC z-score.
    min_effective_n:
        Minimum effective number of names for eligibility.
    use_hard_gate:
        If True, also require existing ``coherence_gate`` from Level 3.
        If False, use only valid APC and effective-N eligibility.
    winsor:
        Winsorization level for z-scores.
    output_dir:
        If supplied, save scores, returns, metrics and weights to this folder.

    Returns
    -------
    dict with scores, returns, metrics and per-strategy results.
    """
    if "theme_returns" not in results_l3:
        raise KeyError("results_l3 must contain 'theme_returns'")

    theme_returns = results_l3["theme_returns"].copy()
    cfg = config or StrategyConfig(
        mode="long_only",
        top_n=5,
        max_theme_weight=0.20,
        use_news=False,
        use_governance=False,
        use_leadlag=False,
        use_valuation=False,
        use_crowding=False,
    )
    # Ensure overlay flags do not affect score-variant backtests.
    cfg = replace(cfg, use_news=False, use_governance=False, use_leadlag=False, use_valuation=False)

    scores = build_apc_variant_scores(
        results_l3,
        apc_z_window=apc_z_window,
        apc_z_min_periods=apc_z_min_periods,
        min_effective_n=min_effective_n,
        use_hard_gate=use_hard_gate,
        winsor=winsor,
    )

    strategy_score_map = {
        "Pure_Coherence": scores["pure_coherence_score"],
        "Coherence_Gated_Momentum": scores["coherence_gated_momentum_score"],
        "APC_Activation_Positive_Momentum": scores["apc_activation_momentum_positive_score"],
    }

    per_strategy = {}
    returns = pd.DataFrame(index=theme_returns.index)
    metrics = {}

    for name, score in strategy_score_map.items():
        res = backtest_score_variant(
            theme_returns=theme_returns,
            score=score,
            config=cfg,
            gate=scores["eligibility"],
        )
        per_strategy[name] = res
        returns[name] = res["returns"]
        metrics[name] = res["metrics"]

    metrics_df = pd.concat(metrics, axis=1).T

    out = {
        "scores": scores,
        "returns": returns,
        "metrics": metrics_df,
        "per_strategy": per_strategy,
        "config": cfg,
    }

    if output_dir is not None:
        os.makedirs(output_dir, exist_ok=True)
        for key, df in scores.items():
            df.to_csv(os.path.join(output_dir, f"{key}.csv"), index=True)
        returns.to_csv(os.path.join(output_dir, "apc_variant_returns.csv"), index=True)
        metrics_df.to_csv(os.path.join(output_dir, "apc_variant_metrics.csv"), index=True)
        for name, res in per_strategy.items():
            res["weights_rebalance"].to_csv(os.path.join(output_dir, f"{name}_weights_rebalance.csv"), index=True)
            res["weights_daily"].to_csv(os.path.join(output_dir, f"{name}_weights_daily.csv"), index=True)
            res["details"].to_csv(os.path.join(output_dir, f"{name}_backtest_details.csv"), index=True)

    return out


## 3. Residual RRG implementation

このセルは residual RRG の計算・戦略化・チャート化ロジックです。

In [ ]:
"""Residual RRG utilities for theme-basket rotation.

This module is designed as an add-on to ``theme_basket_strategy.py``.
It uses ``results_l3['theme_residual_returns']`` as the preferred input
and creates a Relative Rotation Graph (RRG)-like representation for themes.

Key idea
--------
Classic RRG is built from relative strength and the momentum of relative
strength. Here we build those quantities from factor-residualized theme
returns, so the chart emphasizes theme-specific residual leadership rather
than market / style / sector beta.

Dependencies: pandas, numpy, matplotlib, and theme_basket_strategy.py.
"""
import os
from dataclasses import replace
from typing import Dict, Iterable, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd

# Core strategy classes/functions are defined above in this notebook.


ArrayLikeFrame = Union[pd.DataFrame, pd.Series]


def rolling_time_series_zscore(
    x: pd.DataFrame,
    window: int = 252,
    min_periods: Optional[int] = None,
    std_floor: float = 1e-8,
    winsor: Optional[float] = 5.0,
) -> pd.DataFrame:
    """Rolling time-series z-score for each column.

    Notes
    -----
    The signal at date t uses information up to t. Existing backtest helpers
    in ``theme_basket_strategy.py`` lag weights by one day, so this is safe
    for a daily close-to-close backtest if signals are formed after the close.
    """
    if min_periods is None:
        min_periods = max(20, window // 2)
    y = x.copy().sort_index().replace([np.inf, -np.inf], np.nan)
    mu = y.rolling(window=window, min_periods=min_periods).mean()
    sd = y.rolling(window=window, min_periods=min_periods).std(ddof=0)
    z = (y - mu) / sd.clip(lower=std_floor)
    if winsor is not None:
        z = z.clip(-winsor, winsor)
    return z


def _as_aligned_series(x: pd.Series, index: pd.Index, name: str = "benchmark") -> pd.Series:
    s = x.copy().sort_index().replace([np.inf, -np.inf], np.nan)
    s.name = name
    return s.reindex(index).astype(float)


def residual_relative_strength(
    residual_returns: pd.DataFrame,
    benchmark_returns: Optional[pd.Series] = None,
    window: int = 126,
    min_periods: Optional[int] = None,
    peer_relative: bool = True,
    vol_adjust: bool = True,
    vol_floor: float = 1e-4,
) -> pd.DataFrame:
    """Compute residual relative strength for each theme.

    Parameters
    ----------
    residual_returns:
        Date x theme residual return matrix.
    benchmark_returns:
        Optional benchmark return series. If supplied, each theme residual is
        measured relative to this benchmark. If omitted and ``peer_relative``
        is True, the benchmark is the cross-sectional mean residual return.
    window:
        Lookback window for cumulative relative residual return.
    min_periods:
        Minimum observations. If omitted, max(20, window//2).
    peer_relative:
        If True and benchmark_returns is None, subtract cross-sectional mean.
        If False and no benchmark is supplied, use residual_returns directly.
    vol_adjust:
        If True, divide cumulative relative residual return by rolling
        residual-relative volatility, creating a t-stat-like relative strength.
    vol_floor:
        Lower bound for volatility denominator.

    Returns
    -------
    DataFrame of residual relative strength, date x theme.
    """
    if min_periods is None:
        min_periods = max(20, window // 2)

    ret = residual_returns.copy().sort_index().replace([np.inf, -np.inf], np.nan)

    if benchmark_returns is not None:
        b = _as_aligned_series(benchmark_returns, ret.index)
        rel = ret.sub(b, axis=0)
    elif peer_relative:
        peer = ret.mean(axis=1, skipna=True)
        rel = ret.sub(peer, axis=0)
    else:
        rel = ret

    rs = rel.rolling(window=window, min_periods=min_periods).sum()
    if vol_adjust:
        vol = rel.rolling(window=window, min_periods=min_periods).std(ddof=0)
        rs = rs / vol.clip(lower=vol_floor)
    return rs.replace([np.inf, -np.inf], np.nan)


def classify_rrg_quadrants(
    rs_ratio: pd.DataFrame,
    rs_momentum: pd.DataFrame,
    center: float = 100.0,
) -> pd.DataFrame:
    """Classify each theme/date into RRG quadrants."""
    x = rs_ratio.reindex_like(rs_momentum)
    y = rs_momentum
    quad = pd.DataFrame(np.nan, index=y.index, columns=y.columns, dtype=object)
    valid = x.notna() & y.notna()
    quad[valid & (x >= center) & (y >= center)] = "Leading"
    quad[valid & (x >= center) & (y < center)] = "Weakening"
    quad[valid & (x < center) & (y < center)] = "Lagging"
    quad[valid & (x < center) & (y >= center)] = "Improving"
    return quad


def _angle_diff(theta: pd.DataFrame) -> pd.DataFrame:
    """Wrapped one-period angular difference in radians."""
    d = theta - theta.shift(1)
    return ((d + np.pi) % (2 * np.pi)) - np.pi


def make_residual_rrg(
    residual_returns: pd.DataFrame,
    benchmark_returns: Optional[pd.Series] = None,
    rs_window: int = 126,
    rs_min_periods: Optional[int] = None,
    z_window: int = 252,
    z_min_periods: Optional[int] = None,
    momentum_lag: int = 21,
    smoothing_span: int = 10,
    peer_relative: bool = True,
    vol_adjust: bool = True,
    center: float = 100.0,
    scale: float = 10.0,
    winsor: Optional[float] = 5.0,
) -> Dict[str, pd.DataFrame]:
    """Build residual-RRG coordinates and derived features.

    Output fields
    -------------
    - ``residual_relative_strength``: rolling relative residual strength.
    - ``rs_ratio_z``: time-series z-score of smoothed relative strength.
    - ``rs_momentum_z``: time-series z-score of momentum of relative strength.
    - ``rs_ratio``: RRG x-axis, centered at 100.
    - ``rs_momentum``: RRG y-axis, centered at 100.
    - ``quadrant``: Leading / Weakening / Lagging / Improving.
    - ``distance``: distance from the RRG center.
    - ``angle``: polar angle around the RRG center, radians.
    - ``rotation_speed``: wrapped one-period angle change, radians.
    - ``rrg_score``: simple strategy score; positive in Improving/Leading.
    """
    ret = residual_returns.copy().sort_index().replace([np.inf, -np.inf], np.nan)

    rs_raw = residual_relative_strength(
        ret,
        benchmark_returns=benchmark_returns,
        window=rs_window,
        min_periods=rs_min_periods,
        peer_relative=peer_relative,
        vol_adjust=vol_adjust,
    )

    if smoothing_span and smoothing_span > 1:
        rs_smooth = rs_raw.ewm(span=smoothing_span, adjust=False, min_periods=max(2, smoothing_span // 2)).mean()
    else:
        rs_smooth = rs_raw

    rs_ratio_z = rolling_time_series_zscore(
        rs_smooth,
        window=z_window,
        min_periods=z_min_periods,
        winsor=winsor,
    )

    rs_mom_raw = rs_smooth - rs_smooth.shift(momentum_lag)
    rs_momentum_z = rolling_time_series_zscore(
        rs_mom_raw,
        window=z_window,
        min_periods=z_min_periods,
        winsor=winsor,
    )

    rs_ratio = center + scale * rs_ratio_z
    rs_momentum = center + scale * rs_momentum_z
    quadrant = classify_rrg_quadrants(rs_ratio, rs_momentum, center=center)

    dx = rs_ratio - center
    dy = rs_momentum - center
    distance = np.sqrt(dx.pow(2) + dy.pow(2))
    angle = pd.DataFrame(np.arctan2(dy.to_numpy(dtype=float), dx.to_numpy(dtype=float)), index=ret.index, columns=ret.columns)
    rotation_speed = _angle_diff(angle)

    # A transparent RRG strategy score. The quadrant filter is applied later.
    # This keeps the raw score available for diagnostics.
    rrg_score = 0.55 * rs_ratio_z + 0.45 * rs_momentum_z

    latest_rows = []
    if len(ret.index) > 0:
        last_valid_date = rs_ratio.dropna(how="all").index.max()
        if pd.notna(last_valid_date):
            for theme in ret.columns:
                latest_rows.append({
                    "date": last_valid_date,
                    "theme": theme,
                    "rs_ratio": rs_ratio.at[last_valid_date, theme] if theme in rs_ratio.columns else np.nan,
                    "rs_momentum": rs_momentum.at[last_valid_date, theme] if theme in rs_momentum.columns else np.nan,
                    "quadrant": quadrant.at[last_valid_date, theme] if theme in quadrant.columns else np.nan,
                    "distance": distance.at[last_valid_date, theme] if theme in distance.columns else np.nan,
                    "angle": angle.at[last_valid_date, theme] if theme in angle.columns else np.nan,
                    "rotation_speed": rotation_speed.at[last_valid_date, theme] if theme in rotation_speed.columns else np.nan,
                    "rrg_score": rrg_score.at[last_valid_date, theme] if theme in rrg_score.columns else np.nan,
                })
    latest_summary = pd.DataFrame(latest_rows)
    if not latest_summary.empty:
        latest_summary = latest_summary.sort_values(["quadrant", "rrg_score"], ascending=[True, False])

    return {
        "residual_relative_strength": rs_raw,
        "residual_relative_strength_smooth": rs_smooth,
        "rs_ratio_z": rs_ratio_z,
        "rs_momentum_z": rs_momentum_z,
        "rs_ratio": rs_ratio,
        "rs_momentum": rs_momentum,
        "quadrant": quadrant,
        "distance": distance,
        "angle": angle,
        "rotation_speed": rotation_speed,
        "rrg_score": rrg_score,
        "latest_summary": latest_summary,
    }


def _rrg_eligibility_from_results(
    results_l3: Dict[str, object],
    index: pd.Index,
    columns: Sequence[str],
    min_effective_n: float = 5.0,
    use_hard_gate: bool = True,
) -> pd.DataFrame:
    """Build an eligibility mask from Level-3 coherence outputs if available."""
    eligibility = pd.DataFrame(True, index=index, columns=columns)

    apc = results_l3.get("coherence_apc")
    if isinstance(apc, pd.DataFrame):
        eligibility &= apc.reindex(index).ffill().reindex(columns=columns).notna()

    eff = results_l3.get("coherence_effective_n")
    if isinstance(eff, pd.DataFrame):
        eligibility &= eff.reindex(index).ffill().reindex(columns=columns) >= min_effective_n

    if use_hard_gate:
        gate = results_l3.get("coherence_gate")
        if isinstance(gate, pd.DataFrame):
            eligibility &= gate.reindex(index).ffill().reindex(columns=columns).fillna(False)

    return eligibility.fillna(False)


def build_residual_rrg_scores(
    results_l3: Dict[str, object],
    benchmark_returns: Optional[pd.Series] = None,
    rs_window: int = 126,
    rs_min_periods: Optional[int] = None,
    z_window: int = 252,
    z_min_periods: Optional[int] = None,
    momentum_lag: int = 21,
    smoothing_span: int = 10,
    peer_relative: bool = True,
    vol_adjust: bool = True,
    use_hard_gate: bool = True,
    min_effective_n: float = 5.0,
    allowed_quadrants: Tuple[str, ...] = ("Improving", "Leading"),
    require_positive_momentum: bool = True,
    crowding_penalty: float = 0.0,
    winsor: Optional[float] = 5.0,
) -> Dict[str, object]:
    """Create residual-RRG score matrices from ``results_l3``.

    The preferred signal input is ``results_l3['theme_residual_returns']``.
    If unavailable, the function falls back to ``results_l3['theme_returns']``.
    """
    if "theme_residual_returns" in results_l3 and isinstance(results_l3["theme_residual_returns"], pd.DataFrame):
        residual_returns = results_l3["theme_residual_returns"].copy()
    elif "theme_returns" in results_l3 and isinstance(results_l3["theme_returns"], pd.DataFrame):
        residual_returns = results_l3["theme_returns"].copy()
    else:
        raise KeyError("results_l3 must contain 'theme_residual_returns' or 'theme_returns'.")

    rrg = make_residual_rrg(
        residual_returns=residual_returns,
        benchmark_returns=benchmark_returns,
        rs_window=rs_window,
        rs_min_periods=rs_min_periods,
        z_window=z_window,
        z_min_periods=z_min_periods,
        momentum_lag=momentum_lag,
        smoothing_span=smoothing_span,
        peer_relative=peer_relative,
        vol_adjust=vol_adjust,
        winsor=winsor,
    )

    base_score = rrg["rrg_score"].copy()
    quadrant = rrg["quadrant"]
    eligibility = _rrg_eligibility_from_results(
        results_l3,
        index=base_score.index,
        columns=base_score.columns,
        min_effective_n=min_effective_n,
        use_hard_gate=use_hard_gate,
    )

    if allowed_quadrants:
        qmask = quadrant.isin(list(allowed_quadrants))
    else:
        qmask = pd.DataFrame(True, index=base_score.index, columns=base_score.columns)

    if require_positive_momentum:
        qmask &= rrg["rs_momentum"] > 100.0

    score = base_score.where(eligibility & qmask)

    if crowding_penalty and crowding_penalty > 0:
        crowd = results_l3.get("crowding_score")
        if isinstance(crowd, pd.DataFrame):
            c = crowd.reindex(score.index).ffill().reindex(columns=score.columns).fillna(0.0)
            # Penalize only positive crowding/acceleration. Negative crowding is not rewarded.
            score = score - crowding_penalty * c.clip(lower=0.0)
            score = score.where(eligibility & qmask)

    return {
        "rrg": rrg,
        "eligibility": eligibility,
        "rrg_strategy_score": score,
    }


def run_residual_rrg_strategy(
    results_l3: Dict[str, object],
    config: Optional[StrategyConfig] = None,
    benchmark_returns: Optional[pd.Series] = None,
    rs_window: int = 126,
    rs_min_periods: Optional[int] = None,
    z_window: int = 252,
    z_min_periods: Optional[int] = None,
    momentum_lag: int = 21,
    smoothing_span: int = 10,
    peer_relative: bool = True,
    vol_adjust: bool = True,
    use_hard_gate: bool = True,
    min_effective_n: float = 5.0,
    allowed_quadrants: Tuple[str, ...] = ("Improving", "Leading"),
    require_positive_momentum: bool = True,
    crowding_penalty: float = 0.0,
    winsor: Optional[float] = 5.0,
    output_dir: Optional[str] = None,
) -> Dict[str, object]:
    """Run a residual-RRG theme rotation backtest.

    Signal is residual-based, but P&L is computed on ``theme_returns`` because
    actual portfolio performance includes all realized theme returns.
    """
    if "theme_returns" not in results_l3 or not isinstance(results_l3["theme_returns"], pd.DataFrame):
        raise KeyError("results_l3 must contain 'theme_returns' for backtesting.")
    theme_returns = results_l3["theme_returns"].copy().sort_index()

    cfg = config or StrategyConfig(
        mode="long_only",
        top_n=5,
        max_theme_weight=0.20,
        use_news=False,
        use_governance=False,
        use_leadlag=False,
        use_valuation=False,
        use_crowding=False,
    )
    # Ensure existing overlay flags do not double-count with RRG score.
    cfg = replace(cfg, use_news=False, use_governance=False, use_leadlag=False, use_valuation=False)

    built = build_residual_rrg_scores(
        results_l3=results_l3,
        benchmark_returns=benchmark_returns,
        rs_window=rs_window,
        rs_min_periods=rs_min_periods,
        z_window=z_window,
        z_min_periods=z_min_periods,
        momentum_lag=momentum_lag,
        smoothing_span=smoothing_span,
        peer_relative=peer_relative,
        vol_adjust=vol_adjust,
        use_hard_gate=use_hard_gate,
        min_effective_n=min_effective_n,
        allowed_quadrants=allowed_quadrants,
        require_positive_momentum=require_positive_momentum,
        crowding_penalty=crowding_penalty,
        winsor=winsor,
    )

    score = built["rrg_strategy_score"].reindex(theme_returns.index).reindex(columns=theme_returns.columns)
    eligibility = built["eligibility"].reindex(theme_returns.index).ffill().reindex(columns=theme_returns.columns).fillna(False)

    rebal_dates = make_rebalance_dates(theme_returns.index, cfg.rebalance)
    weights_rebal = construct_theme_weights(
        score.reindex(rebal_dates),
        cfg,
        gates=eligibility.reindex(rebal_dates).fillna(False),
    )
    weights_daily = weights_rebal.reindex(theme_returns.index).ffill().fillna(0.0)

    strat_ret, details = backtest_theme_weights(
        theme_returns=theme_returns,
        weights_daily=weights_daily,
        cost_bps_per_turnover=cfg.cost_bps_per_turnover,
    )
    metrics = performance_metrics(strat_ret, cfg.annualization)

    out = {
        **built,
        "weights_rebalance": weights_rebal,
        "weights_daily": weights_daily,
        "returns": strat_ret,
        "details": details,
        "metrics": metrics,
        "config": cfg,
    }

    if output_dir is not None:
        os.makedirs(output_dir, exist_ok=True)
        rrg = built["rrg"]
        for key, obj in rrg.items():
            if isinstance(obj, pd.DataFrame):
                obj.to_csv(os.path.join(output_dir, f"residual_rrg_{key}.csv"), index=True)
        built["eligibility"].to_csv(os.path.join(output_dir, "residual_rrg_eligibility.csv"), index=True)
        built["rrg_strategy_score"].to_csv(os.path.join(output_dir, "residual_rrg_strategy_score.csv"), index=True)
        weights_rebal.to_csv(os.path.join(output_dir, "Residual_RRG_weights_rebalance.csv"), index=True)
        weights_daily.to_csv(os.path.join(output_dir, "Residual_RRG_weights_daily.csv"), index=True)
        strat_ret.to_csv(os.path.join(output_dir, "residual_rrg_strategy_returns.csv"), header=True)
        details.to_csv(os.path.join(output_dir, "residual_rrg_backtest_details.csv"), index=True)
        metrics.to_csv(os.path.join(output_dir, "residual_rrg_metrics.csv"), header=True)

    return out


def plot_residual_rrg(
    rrg: Dict[str, pd.DataFrame],
    date: Optional[pd.Timestamp] = None,
    themes: Optional[Sequence[str]] = None,
    tail: int = 12,
    center: float = 100.0,
    figsize: Tuple[float, float] = (10.0, 8.0),
    title: Optional[str] = None,
    output_path: Optional[str] = None,
    show: bool = True,
):
    """Plot residual-RRG coordinates with short tails.

    Parameters
    ----------
    rrg:
        Output ``out['rrg']`` from ``make_residual_rrg`` or
        ``run_residual_rrg_strategy``.
    date:
        Date to plot. If omitted, uses the last date with any RS-Ratio.
    themes:
        Optional subset/order of themes.
    tail:
        Number of historical observations to draw for each theme.
    output_path:
        If supplied, save the chart to this path.
    show:
        If True, call ``plt.show()``.
    """
    import matplotlib.pyplot as plt

    x = rrg["rs_ratio"].copy()
    y = rrg["rs_momentum"].copy()
    if date is None:
        date = x.dropna(how="all").index.max()
    date = pd.Timestamp(date)
    if date not in x.index:
        pos = x.index.searchsorted(date, side="right") - 1
        if pos < 0:
            raise ValueError("No RRG data available on or before the requested date.")
        date = x.index[pos]

    if themes is None:
        row = x.loc[date].dropna()
        themes = list(row.index)
    else:
        themes = [str(t) for t in themes if str(t) in x.columns]

    fig, ax = plt.subplots(figsize=figsize)
    ax.axhline(center, linewidth=1)
    ax.axvline(center, linewidth=1)
    ax.set_xlabel("Residual RS-Ratio")
    ax.set_ylabel("Residual RS-Momentum")
    ax.set_title(title or f"Residual RRG as of {date.date()}")

    # Quadrant labels.
    xmin = np.nanpercentile(x.loc[:date, themes].to_numpy(dtype=float), 5) if themes else center - 10
    xmax = np.nanpercentile(x.loc[:date, themes].to_numpy(dtype=float), 95) if themes else center + 10
    ymin = np.nanpercentile(y.loc[:date, themes].to_numpy(dtype=float), 5) if themes else center - 10
    ymax = np.nanpercentile(y.loc[:date, themes].to_numpy(dtype=float), 95) if themes else center + 10
    pad_x = max(5.0, 0.15 * (xmax - xmin if np.isfinite(xmax - xmin) else 10.0))
    pad_y = max(5.0, 0.15 * (ymax - ymin if np.isfinite(ymax - ymin) else 10.0))
    ax.set_xlim(min(center - 5, xmin - pad_x), max(center + 5, xmax + pad_x))
    ax.set_ylim(min(center - 5, ymin - pad_y), max(center + 5, ymax + pad_y))
    ax.text(center + pad_x * 0.25, center + pad_y * 0.25, "Leading", fontsize=10)
    ax.text(center + pad_x * 0.25, center - pad_y * 0.75, "Weakening", fontsize=10)
    ax.text(center - pad_x * 1.25, center - pad_y * 0.75, "Lagging", fontsize=10)
    ax.text(center - pad_x * 1.25, center + pad_y * 0.25, "Improving", fontsize=10)

    end_loc = x.index.get_loc(date)
    if not isinstance(end_loc, (int, np.integer)):
        end_loc = int(np.asarray(end_loc).ravel()[-1])
    start_loc = max(0, end_loc - max(1, tail) + 1)
    idx_tail = x.index[start_loc : end_loc + 1]

    for theme in themes:
        xt = x.loc[idx_tail, theme].dropna()
        yt = y.loc[idx_tail, theme].reindex(xt.index).dropna()
        common = xt.index.intersection(yt.index)
        if len(common) == 0:
            continue
        ax.plot(xt.loc[common], yt.loc[common], marker="o", linewidth=1)
        ax.scatter([x.at[date, theme]], [y.at[date, theme]], s=40)
        ax.annotate(theme, (x.at[date, theme], y.at[date, theme]), textcoords="offset points", xytext=(4, 4), fontsize=9)

    fig.tight_layout()
    if output_path is not None:
        fig.savefig(output_path, dpi=150, bbox_inches="tight")
    if show:
        plt.show()
    return fig, ax


## 4. パス・入力設定

実データで実行する場合は `USE_SAMPLE_DATA=False` にし、各CSVパスを指定してください。

In [ ]:
# True: サンプルデータを生成して全セルを動作確認
# False: 手元のCSVパスを使う
USE_SAMPLE_DATA = True

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "sample_theme_data"
OUTPUT_ROOT = BASE_DIR / "theme_strategy_outputs"
OUTPUT_ROOT.mkdir(exist_ok=True)

# 実データ利用時はここを書き換えてください。
PATH_THEME_CSV = BASE_DIR / "theme_prices.csv"             # 必須: テーマ価格 or リターン
PATH_FACTOR_CSV = BASE_DIR / "factor_returns.csv"          # 任意: TOPIX/業種/スタイル等のリターン
PATH_CONSTITUENTS_CSV = BASE_DIR / "constituents.csv"      # 任意: PIT構成銘柄・ウェイト
PATH_STOCK_CSV = BASE_DIR / "stock_prices.csv"             # 任意: 個別株価格 or リターン
PATH_NEWS_CSV = BASE_DIR / "news_score.csv"                # 任意: テーマ別ニュース/attentionスコア
PATH_GOVERNANCE_CSV = BASE_DIR / "governance_score.csv"    # 任意: テーマ別PBR/ガバナンススコア
PATH_TURNOVER_CSV = BASE_DIR / "theme_turnover.csv"        # 任意: テーマ別回転率/出来高

# 入力CSVが価格かリターンかを指定します。
# price: 価格指数からpct_changeでリターン化
# return: すでにリターン
# auto: 値の大きさから簡易推定
THEME_INPUT_KIND = "price"
STOCK_INPUT_KIND = "price"

## 5. サンプルデータ生成

`USE_SAMPLE_DATA=True` の場合のみ、テーマ価格、ファクターリターン、構成銘柄、個別株価格、ニューススコア、ガバナンススコア、回転率を生成します。

In [ ]:
if USE_SAMPLE_DATA:
    np.random.seed(42)
    DATA_DIR.mkdir(exist_ok=True)

    dates = pd.bdate_range("2022-01-03", "2025-12-31")
    n = len(dates)
    themes = ["AI", "Semiconductor", "Defense", "LowPBR", "Governance", "Unprofitable"]

    # ファクターリターン
    factors = pd.DataFrame(
        {
            "TOPIX": np.random.normal(0.00015, 0.0090, n),
            "Size": np.random.normal(0.00002, 0.0050, n),
            "Value": np.random.normal(0.00003, 0.0055, n),
            "Momentum": np.random.normal(0.00004, 0.0050, n),
            "Rate10Y": np.random.normal(0.00000, 0.0020, n),
            "SOX": np.random.normal(0.00020, 0.0120, n),
        },
        index=dates,
    )

    # テーマ固有ショック。一部テーマにトレンドを入れる。
    theme_shocks = pd.DataFrame(index=dates, columns=themes, dtype=float)
    for th in themes:
        theme_shocks[th] = np.random.normal(0, 0.006, n)
    theme_shocks.loc["2023":, "AI"] += 0.00045
    theme_shocks.loc["2023":, "Semiconductor"] += 0.00035
    theme_shocks.loc["2024":, "Defense"] += 0.00030
    theme_shocks.loc["2022":, "LowPBR"] += 0.00018
    theme_shocks.loc["2022":, "Governance"] += 0.00020
    theme_shocks.loc["2024":, "Unprofitable"] -= 0.00025

    # 個別株をテーマごとに8銘柄生成
    stock_returns = pd.DataFrame(index=dates)
    constituents_rows = []
    for th in themes:
        for j in range(8):
            ticker = f"{th[:4].upper()}_{j:02d}"
            beta_topix = np.random.uniform(0.6, 1.3)
            beta_size = np.random.uniform(-0.4, 0.5)
            beta_value = np.random.uniform(-0.3, 0.6)
            beta_mom = np.random.uniform(-0.2, 0.5)
            beta_rate = np.random.uniform(-0.2, 0.4)
            beta_sox = np.random.uniform(0.0, 0.5) if th in ["AI", "Semiconductor"] else np.random.uniform(-0.05, 0.1)
            idio = np.random.normal(0, 0.010, n)
            stock_returns[ticker] = (
                beta_topix * factors["TOPIX"]
                + beta_size * factors["Size"]
                + beta_value * factors["Value"]
                + beta_mom * factors["Momentum"]
                + beta_rate * factors["Rate10Y"]
                + beta_sox * factors["SOX"]
                + theme_shocks[th]
                + idio
            )
            constituents_rows.append(
                {
                    "date": pd.Timestamp("2020-01-01"),
                    "theme": th,
                    "ticker": ticker,
                    "weight": 1.0 / 8.0,
                    "purity": np.random.uniform(0.70, 1.00),
                    "liquidity_adj": np.random.uniform(0.85, 1.00),
                }
            )

    stock_prices = (1 + stock_returns).cumprod() * 100.0
    constituents = pd.DataFrame(constituents_rows)

    # テーマ価格は構成銘柄リターンの等ウェイト合成
    theme_returns = pd.DataFrame(index=dates)
    for th in themes:
        tickers = constituents.loc[constituents["theme"] == th, "ticker"].tolist()
        theme_returns[th] = stock_returns[tickers].mean(axis=1)
    theme_prices = (1 + theme_returns).cumprod() * 100.0

    # ニュース・ガバナンス・回転率スコアのサンプル
    try:
        month_ends = theme_prices.resample("ME").last().index
    except ValueError:
        month_ends = theme_prices.resample("M").last().index

    news_score = pd.DataFrame(np.random.normal(0, 1, (len(month_ends), len(themes))), index=month_ends, columns=themes)
    news_score.loc["2023":, "AI"] += 1.0
    news_score.loc["2023":, "Semiconductor"] += 0.8
    news_score.loc["2024":, "Defense"] += 0.7

    governance_score = pd.DataFrame(0.0, index=month_ends, columns=themes)
    governance_score["LowPBR"] = np.linspace(0.2, 1.5, len(month_ends)) + np.random.normal(0, 0.2, len(month_ends))
    governance_score["Governance"] = np.linspace(0.3, 1.3, len(month_ends)) + np.random.normal(0, 0.2, len(month_ends))

    turnover = pd.DataFrame(np.random.lognormal(mean=1.0, sigma=0.25, size=(n, len(themes))), index=dates, columns=themes)
    turnover.loc["2023":, "AI"] *= 1.5
    turnover.loc["2023":, "Semiconductor"] *= 1.3

    PATH_THEME_CSV = DATA_DIR / "theme_prices.csv"
    PATH_FACTOR_CSV = DATA_DIR / "factor_returns.csv"
    PATH_CONSTITUENTS_CSV = DATA_DIR / "constituents.csv"
    PATH_STOCK_CSV = DATA_DIR / "stock_prices.csv"
    PATH_NEWS_CSV = DATA_DIR / "news_score.csv"
    PATH_GOVERNANCE_CSV = DATA_DIR / "governance_score.csv"
    PATH_TURNOVER_CSV = DATA_DIR / "theme_turnover.csv"

    theme_prices.to_csv(PATH_THEME_CSV, index_label="date")
    factors.to_csv(PATH_FACTOR_CSV, index_label="date")
    constituents.to_csv(PATH_CONSTITUENTS_CSV, index=False)
    stock_prices.to_csv(PATH_STOCK_CSV, index_label="date")
    news_score.to_csv(PATH_NEWS_CSV, index_label="date")
    governance_score.to_csv(PATH_GOVERNANCE_CSV, index_label="date")
    turnover.to_csv(PATH_TURNOVER_CSV, index_label="date")

    print("Sample data generated:", DATA_DIR)
else:
    print("USE_SAMPLE_DATA=False: 手元のCSVパスを使います。")

## 6. データロード

In [ ]:
def maybe_read_matrix(path):
    path = Path(path) if path is not None else None
    if path is None or not path.exists():
        return None
    return read_matrix_csv(str(path))

if not Path(PATH_THEME_CSV).exists():
    raise FileNotFoundError(f"テーマ価格CSVが見つかりません: {PATH_THEME_CSV}")

theme_data = read_matrix_csv(str(PATH_THEME_CSV))
factor_returns = maybe_read_matrix(PATH_FACTOR_CSV)
stock_data = maybe_read_matrix(PATH_STOCK_CSV)
news_score = maybe_read_matrix(PATH_NEWS_CSV)
governance_score = maybe_read_matrix(PATH_GOVERNANCE_CSV)
turnover_data = maybe_read_matrix(PATH_TURNOVER_CSV)

constituents = None
if Path(PATH_CONSTITUENTS_CSV).exists():
    constituents = read_constituents_csv(str(PATH_CONSTITUENTS_CSV))

print("theme_data:", theme_data.shape)
print("factor_returns:", None if factor_returns is None else factor_returns.shape)
print("stock_data:", None if stock_data is None else stock_data.shape)
print("constituents:", None if constituents is None else constituents.shape)
print("news_score:", None if news_score is None else news_score.shape)
print("governance_score:", None if governance_score is None else governance_score.shape)
print("turnover_data:", None if turnover_data is None else turnover_data.shape)

display(theme_data.tail())

## 7. Level 1: MVP — 生テーマモメンタム

In [ ]:
cfg_l1 = StrategyConfig(
    theme_input_kind=THEME_INPUT_KIND,
    mode="long_only",
    rebalance="ME",
    top_n=5,
    max_theme_weight=0.25,
    cost_bps_per_turnover=5.0,
    residualize_themes=False,
    use_coherence_gate=False,
    use_news=False,
    use_governance=False,
    use_leadlag=False,
    use_valuation=False,
    use_crowding=True,
)

strategy_l1 = ThemeBasketStrategy(theme_data=theme_data, config=cfg_l1)
results_l1 = strategy_l1.compute()
strategy_l1.save_results(str(OUTPUT_ROOT / "level1_mvp"))

print("Level 1 metrics")
display(results_l1["metrics"].to_frame("Level1_MVP").T)
display(results_l1["theme_weights_rebalance"].tail())

## 8. Level 2: Factor-adjusted — テーマ残差モメンタム

In [ ]:
cfg_l2 = StrategyConfig(
    theme_input_kind=THEME_INPUT_KIND,
    mode="long_only",
    rebalance="ME",
    top_n=5,
    max_theme_weight=0.25,
    cost_bps_per_turnover=5.0,
    residualize_themes=True,
    rolling_beta_window=252,
    rolling_beta_min_obs=126,
    use_coherence_gate=False,
    use_news=False,
    use_governance=False,
    use_leadlag=False,
    use_valuation=False,
    use_crowding=True,
)

strategy_l2 = ThemeBasketStrategy(
    theme_data=theme_data,
    config=cfg_l2,
    factor_returns=factor_returns,
)
results_l2 = strategy_l2.compute()
strategy_l2.save_results(str(OUTPUT_ROOT / "level2_factor_adjusted"))

print("Level 2 metrics")
display(results_l2["metrics"].to_frame("Level2_FactorAdjusted").T)
display(results_l2["final_score_gated"].tail())

## 9. Level 3: Coherence-gated — 個別銘柄残差/APC/eigenshare

In [ ]:
if constituents is None or stock_data is None:
    print("Level 3 skipped: constituents または stock_data がありません。")
    results_l3 = None
    cfg_l3 = None
else:
    cfg_l3 = StrategyConfig(
        theme_input_kind=THEME_INPUT_KIND,
        stock_input_kind=STOCK_INPUT_KIND,
        mode="long_only",
        rebalance="ME",
        top_n=5,
        max_theme_weight=0.25,
        cost_bps_per_turnover=5.0,
        residualize_themes=True,
        # サンプル高速化のためFalse。実データで個別株もファクター残差化する場合はTrue。
        residualize_stocks=False,
        rolling_beta_window=252,
        rolling_beta_min_obs=126,
        use_coherence_gate=True,
        coherence_window=126,
        coherence_min_names=8,
        coherence_min_effective_names=5.0,
        coherence_apc_threshold=0.00,
        use_news=False,
        use_governance=False,
        use_leadlag=False,
        use_valuation=False,
        use_crowding=True,
    )

    strategy_l3 = ThemeBasketStrategy(
        theme_data=theme_data,
        config=cfg_l3,
        factor_returns=factor_returns,
        constituents=constituents,
        stock_data=stock_data,
    )
    results_l3 = strategy_l3.compute()
    strategy_l3.save_results(str(OUTPUT_ROOT / "level3_coherence_gated"))

    print("Level 3 metrics")
    display(results_l3["metrics"].to_frame("Level3_CoherenceGated").T)
    print("Coherence APC tail")
    display(results_l3["coherence_apc"].tail())
    print("Coherence gate tail")
    display(results_l3["coherence_gate"].tail())

## 10. Level 4: Full overlay — ニュース、ガバナンス/PBR、回転率crowding

In [ ]:
use_coherence_l4 = (constituents is not None) and (stock_data is not None)

cfg_l4 = StrategyConfig(
    theme_input_kind=THEME_INPUT_KIND,
    stock_input_kind=STOCK_INPUT_KIND,
    mode="long_only",
    rebalance="ME",
    top_n=5,
    max_theme_weight=0.25,
    cost_bps_per_turnover=5.0,
    residualize_themes=True,
    residualize_stocks=False,
    rolling_beta_window=252,
    rolling_beta_min_obs=126,
    use_coherence_gate=use_coherence_l4,
    coherence_window=126,
    coherence_min_names=8,
    coherence_min_effective_names=5.0,
    coherence_apc_threshold=0.00,
    use_news=True,
    use_governance=True,
    use_leadlag=False,
    use_valuation=False,
    use_crowding=True,
    w_momentum=0.45,
    w_news=0.20,
    w_governance=0.20,
    w_leadlag=0.10,
    w_valuation=0.05,
    w_crowding_penalty=0.25,
)

strategy_l4 = ThemeBasketStrategy(
    theme_data=theme_data,
    config=cfg_l4,
    factor_returns=factor_returns,
    constituents=constituents,
    stock_data=stock_data,
    news_score=news_score,
    governance_score=governance_score,
    turnover_data=turnover_data,
)
results_l4 = strategy_l4.compute()
strategy_l4.save_results(str(OUTPUT_ROOT / "level4_full_overlay"))

print("Level 4 metrics")
display(results_l4["metrics"].to_frame("Level4_FullOverlay").T)
print("Final score tail")
display(results_l4["final_score_gated"].tail())
print("Rebalance weights tail")
display(results_l4["theme_weights_rebalance"].tail())

## 11. Level 1–4 結果比較

In [ ]:
metrics = {
    "Level1_MVP": results_l1["metrics"],
    "Level2_FactorAdjusted": results_l2["metrics"],
}
if results_l3 is not None:
    metrics["Level3_CoherenceGated"] = results_l3["metrics"]
metrics["Level4_FullOverlay"] = results_l4["metrics"]

metrics_df = pd.concat(metrics, axis=1).T
display(metrics_df)

nav = pd.DataFrame(index=results_l1["strategy_returns"].index)
nav["Level1_MVP"] = (1 + results_l1["strategy_returns"]).cumprod()
nav["Level2_FactorAdjusted"] = (1 + results_l2["strategy_returns"]).cumprod()
if results_l3 is not None:
    nav["Level3_CoherenceGated"] = (1 + results_l3["strategy_returns"]).cumprod()
nav["Level4_FullOverlay"] = (1 + results_l4["strategy_returns"]).cumprod()

ax = nav.plot(figsize=(12, 5), title="Strategy NAV by execution level")
ax.set_ylabel("NAV")
plt.show()

## 12. APC strategy variants

Level 3 の出力を使い、APC activation、coherence-gated momentum、APC activation + positive momentum を比較します。

In [ ]:
if results_l3 is None:
    print("Level 3 が未実行のため、APC strategy variants はスキップします。")
    apc_variants = None
else:
    cfg_apc_variants = replace(
        cfg_l3,
        mode="long_only",
        top_n=5,
        max_theme_weight=0.20,
        use_news=False,
        use_governance=False,
        use_leadlag=False,
        use_valuation=False,
        use_crowding=False,
    )

    apc_variants = run_apc_strategy_variants(
        results_l3=results_l3,
        config=cfg_apc_variants,
        apc_z_window=252,
        apc_z_min_periods=126,
        min_effective_n=5.0,
        use_hard_gate=True,
        winsor=5.0,
        output_dir=str(OUTPUT_ROOT / "apc_variants"),
    )

    display(apc_variants["metrics"])

    apc_ts_z = apc_variants["scores"]["apc_ts_z"]
    apc_ts_cs_z = apc_variants["scores"]["apc_ts_cs_z"]
    latest_date = apc_ts_cs_z.dropna(how="all").index[-1]
    apc_rank_latest = pd.DataFrame({
        "apc_ts_z": apc_ts_z.loc[latest_date],
        "apc_ts_cs_z": apc_ts_cs_z.loc[latest_date],
        "momentum_score": results_l3["momentum_score"].loc[latest_date],
        "coherence_gate": results_l3["coherence_gate"].loc[latest_date],
    }).sort_values("apc_ts_cs_z", ascending=False)
    display(apc_rank_latest.head(20))

    apc_nav = (1 + apc_variants["returns"].fillna(0)).cumprod()
    ax = apc_nav.plot(figsize=(12, 5), title="APC Strategy Variants NAV")
    ax.set_ylabel("NAV")
    plt.show()

## 13. Residual RRG strategy

`results_l3["theme_residual_returns"]` を優先して使います。Level 3 がない場合は Level 2 のテーマ残差リターンにフォールバックします。  
シグナルは残差リターンで作り、P&Lは実際の `theme_returns` で評価します。

In [ ]:
# RRG入力を決定。APC/coherence gateを使いたい場合はLevel 3が必要です。
if results_l3 is not None:
    rrg_input_results = results_l3
    base_cfg_for_rrg = cfg_l3
    use_hard_gate_rrg = True
else:
    print("Level 3 がないため、Level 2 の残差テーマリターンでResidual RRGを実行します。coherence gateは使いません。")
    rrg_input_results = results_l2
    base_cfg_for_rrg = cfg_l2
    use_hard_gate_rrg = False

cfg_rrg = replace(
    base_cfg_for_rrg,
    mode="long_only",
    top_n=5,
    max_theme_weight=0.20,
    use_news=False,
    use_governance=False,
    use_leadlag=False,
    use_valuation=False,
    use_crowding=False,
)

rrg_res = run_residual_rrg_strategy(
    results_l3=rrg_input_results,
    config=cfg_rrg,
    benchmark_returns=None,          # Noneなら peer-relative。市場ベンチを入れる場合はpd.Seriesを渡す。
    rs_window=126,                   # 約6か月の残差相対強度
    z_window=252,                    # 約1年で時系列zスコア化
    momentum_lag=21,                 # 約1か月のRRGモメンタム
    smoothing_span=10,               # 軌跡のノイズ低減
    peer_relative=True,
    vol_adjust=True,
    use_hard_gate=use_hard_gate_rrg,
    min_effective_n=5.0,
    allowed_quadrants=("Improving", "Leading"),
    require_positive_momentum=True,
    crowding_penalty=0.0,
    winsor=5.0,
    output_dir=str(OUTPUT_ROOT / "residual_rrg"),
)

print("Residual RRG metrics")
display(rrg_res["metrics"].to_frame("Residual_RRG").T)

print("Latest Residual RRG summary")
display(rrg_res["rrg"]["latest_summary"])

print("Latest rebalance weights")
display(rrg_res["weights_rebalance"].tail())

## 14. Residual RRG チャート

In [ ]:
fig, ax = plot_residual_rrg(
    rrg_res["rrg"],
    tail=12,
    output_path=str(OUTPUT_ROOT / "residual_rrg" / "residual_rrg_latest.png"),
    show=True,
)

## 15. Residual RRG を含む最終NAV比較

In [ ]:
nav_all = nav.copy()
nav_all["Residual_RRG"] = (1 + rrg_res["returns"].fillna(0)).cumprod()

if apc_variants is not None:
    for c in apc_variants["returns"].columns:
        nav_all[f"APC_{c}"] = (1 + apc_variants["returns"][c].fillna(0)).cumprod()

ax = nav_all.plot(figsize=(13, 6), title="All strategy NAV comparison")
ax.set_ylabel("NAV")
plt.show()

all_metrics = metrics_df.copy()
all_metrics.loc["Residual_RRG"] = rrg_res["metrics"]
if apc_variants is not None:
    apc_metrics_named = apc_variants["metrics"].copy()
    apc_metrics_named.index = [f"APC_{x}" for x in apc_metrics_named.index]
    all_metrics = pd.concat([all_metrics, apc_metrics_named], axis=0)

display(all_metrics)

## 16. 実データ投入時の最小チェックリスト

1. `theme_prices.csv` または `theme_returns.csv` のカラム名と `constituents.csv` の `theme` 名を一致させる。
2. `constituents.csv` は `date, theme, ticker, weight` を必須列にする。可能なら point-in-time で更新する。
3. `stock_prices.csv` または `stock_returns.csv` のカラム名と `constituents.csv` の `ticker` を一致させる。
4. ファクターは価格ではなくリターン系列で渡す。
5. ニュース、attention、ガバナンス、turnover スコアは、利用可能日ベースで作る。
6. 実運用検証では、ADV制約、テーマ間銘柄重複、個別銘柄集中、取引コスト、税・貸株制約を別途入れる。
